# R3: saturación por brillo y auditoría del preprocesamiento

Este cuaderno responde al punto R3 del Academic Editor. **No ejecuta VGG-16 ni recalcula embeddings.**

Realiza cuatro tareas:

1. verifica las 2,527 imágenes originales y los PNG de brillo realmente usados;
2. comprueba que los PNG guardados coincidan píxel por píxel con la transformación original;
3. cuantifica la saturación preexistente, total y nueva para brillo 1.2, 1.4 y 1.6;
4. documenta el orden de transformación y redimensionamiento, incluida la relación de aspecto.

La medida principal será el **porcentaje de píxeles con al menos un canal RGB igual a 255**. También se informa el porcentaje de píxeles que se volvieron saturados únicamente después de aumentar el brillo, para no confundir saturación preexistente con clipping inducido por la transformación.

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

from pathlib import Path
import json
import platform

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import PIL
import torch
import torchvision
from PIL import Image
from torchvision.transforms import functional as F
from tqdm.auto import tqdm

SEED = 2026
np.random.seed(SEED)

print('Python:', platform.python_version())
print('Pillow:', PIL.__version__)
print('PyTorch:', torch.__version__)
print('torchvision:', torchvision.__version__)

## 1. Configuración

Las rutas corresponden al experimento principal. Modifica solamente `BASE_DIR` si la carpeta `2026-0` se encuentra en otra ubicación de Google Drive.

In [ ]:
BASE_DIR = Path('/content/gdrive/MyDrive/2026-0')
ORIGINAL_ROOT = BASE_DIR / 'DatasetOriginalCompleto'
TRANSFORMED_ROOT = BASE_DIR / 'TrashNet_transformaciones_completo_png'
R3_ROOT = BASE_DIR / 'Revision_R3_Confounds_VGG16'
R3_ROOT.mkdir(parents=True, exist_ok=True)

CLASSES = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
EXPECTED_COUNTS = {
    'cardboard': 403,
    'glass': 501,
    'metal': 410,
    'paper': 594,
    'plastic': 482,
    'trash': 137,
}
BRIGHTNESS = {'low': 1.2, 'medium': 1.4, 'high': 1.6}
VALID_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp'}

print('Originales:', ORIGINAL_ROOT)
print('Transformadas:', TRANSFORMED_ROOT)
print('Salida R3:', R3_ROOT)

## 2. Verificación de archivos y dimensiones

TrashNet contiene imágenes de 512×384 píxeles. Se verificará el conteo por clase y que las imágenes de brillo almacenadas conserven esas dimensiones antes de la entrada a VGG-16.

In [ ]:
def list_images(folder):
    return sorted(
        [p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in VALID_EXTENSIONS],
        key=lambda p: p.name.lower()
    )

if not ORIGINAL_ROOT.exists():
    raise FileNotFoundError(f'No existe {ORIGINAL_ROOT}')
if not TRANSFORMED_ROOT.exists():
    raise FileNotFoundError(f'No existe {TRANSFORMED_ROOT}')

files_by_class = {}
dimension_counts = {}

for class_name in CLASSES:
    class_dir = ORIGINAL_ROOT / class_name
    files = list_images(class_dir)
    files_by_class[class_name] = files
    if len(files) != EXPECTED_COUNTS[class_name]:
        raise ValueError(f'{class_name}: {len(files)} imágenes; se esperaban {EXPECTED_COUNTS[class_name]}')
    for path in files:
        with Image.open(path) as image:
            dimension_counts[image.size] = dimension_counts.get(image.size, 0) + 1

print('Conteos por clase:', {c: len(files_by_class[c]) for c in CLASSES})
print('Total:', sum(len(v) for v in files_by_class.values()))
print('Dimensiones originales:', dimension_counts)

if dimension_counts != {(512, 384): 2527}:
    raise ValueError('Las dimensiones originales no coinciden con las documentadas para TrashNet.')

## 3. Medición de clipping

Se distinguen tres cantidades:

- **preexisting**: el píxel ya tenía al menos un canal en 255 antes de aumentar el brillo;
- **after**: el píxel tiene al menos un canal en 255 en el PNG transformado;
- **newly clipped**: el píxel no estaba saturado en el original y sí lo está después de la transformación.

La misma distinción se calcula por componentes RGB y para píxeles completamente blancos.

In [ ]:
rows = []

for class_name in CLASSES:
    for original_path in tqdm(files_by_class[class_name], desc=class_name):
        with Image.open(original_path) as image:
            original_image = image.convert('RGB')
            original = np.asarray(original_image, dtype=np.uint8)

        height, width, channels = original.shape
        n_pixels = height * width
        n_channels = n_pixels * channels

        original_any = np.any(original == 255, axis=2)
        original_all = np.all(original == 255, axis=2)

        for level, factor in BRIGHTNESS.items():
            transformed_path = (
                TRANSFORMED_ROOT / f'brightness_{level}' / class_name /
                f'{original_path.stem}.png'
            )
            if not transformed_path.exists():
                raise FileNotFoundError(f'Falta {transformed_path}')

            with Image.open(transformed_path) as transformed_image:
                stored = np.asarray(transformed_image.convert('RGB'), dtype=np.uint8)

            if stored.shape != original.shape:
                raise ValueError(
                    f'Dimensiones diferentes en {transformed_path}: '
                    f'{stored.shape} frente a {original.shape}'
                )

            regenerated = np.asarray(
                F.adjust_brightness(original_image, brightness_factor=factor),
                dtype=np.uint8
            )
            absolute_difference = np.abs(stored.astype(np.int16) - regenerated.astype(np.int16))

            stored_any = np.any(stored == 255, axis=2)
            stored_all = np.all(stored == 255, axis=2)
            newly_any = stored_any & ~original_any
            newly_all = stored_all & ~original_all
            original_channel = original == 255
            stored_channel = stored == 255
            newly_channel = stored_channel & ~original_channel

            rows.append({
                'class': class_name,
                'original_filename': original_path.name,
                'level': level,
                'brightness_factor': factor,
                'width': width,
                'height': height,
                'n_pixels': n_pixels,
                'n_channels': n_channels,
                'preexisting_any255_count': int(original_any.sum()),
                'after_any255_count': int(stored_any.sum()),
                'newly_any255_count': int(newly_any.sum()),
                'preexisting_all255_count': int(original_all.sum()),
                'after_all255_count': int(stored_all.sum()),
                'newly_all255_count': int(newly_all.sum()),
                'preexisting_channel255_count': int(original_channel.sum()),
                'after_channel255_count': int(stored_channel.sum()),
                'newly_channel255_count': int(newly_channel.sum()),
                'preexisting_any255_proportion': float(original_any.mean()),
                'after_any255_proportion': float(stored_any.mean()),
                'newly_any255_proportion': float(newly_any.mean()),
                'preexisting_all255_proportion': float(original_all.mean()),
                'after_all255_proportion': float(stored_all.mean()),
                'newly_all255_proportion': float(newly_all.mean()),
                'preexisting_channel255_proportion': float(original_channel.mean()),
                'after_channel255_proportion': float(stored_channel.mean()),
                'newly_channel255_proportion': float(newly_channel.mean()),
                'stored_vs_regenerated_max_abs_difference': int(absolute_difference.max()),
                'stored_vs_regenerated_mismatch_proportion': float((absolute_difference > 0).mean()),
            })

individual = pd.DataFrame(rows)
individual.head()

## 4. Resumen global y por imagen

El resumen global pondera cada píxel por igual. También se informa la mediana y el intervalo intercuartílico entre imágenes para mostrar la variabilidad del dataset.

In [ ]:
summary_rows = []

for level, factor in BRIGHTNESS.items():
    group = individual[individual['level'] == level].copy()
    total_pixels = group['n_pixels'].sum()
    total_channels = group['n_channels'].sum()

    q1 = group['newly_any255_proportion'].quantile(0.25)
    median = group['newly_any255_proportion'].median()
    q3 = group['newly_any255_proportion'].quantile(0.75)

    summary_rows.append({
        'level': level,
        'brightness_factor': factor,
        'n_images': len(group),
        'total_pixels': int(total_pixels),
        'preexisting_any255_percent_global': 100 * group['preexisting_any255_count'].sum() / total_pixels,
        'after_any255_percent_global': 100 * group['after_any255_count'].sum() / total_pixels,
        'newly_any255_percent_global': 100 * group['newly_any255_count'].sum() / total_pixels,
        'newly_any255_percent_image_q1': 100 * q1,
        'newly_any255_percent_image_median': 100 * median,
        'newly_any255_percent_image_q3': 100 * q3,
        'after_all255_percent_global': 100 * group['after_all255_count'].sum() / total_pixels,
        'newly_all255_percent_global': 100 * group['newly_all255_count'].sum() / total_pixels,
        'after_channel255_percent_global': 100 * group['after_channel255_count'].sum() / total_channels,
        'newly_channel255_percent_global': 100 * group['newly_channel255_count'].sum() / total_channels,
        'max_stored_vs_regenerated_abs_difference': int(group['stored_vs_regenerated_max_abs_difference'].max()),
        'max_stored_vs_regenerated_mismatch_proportion': float(group['stored_vs_regenerated_mismatch_proportion'].max()),
    })

summary = pd.DataFrame(summary_rows)
summary

In [ ]:
if summary['max_stored_vs_regenerated_abs_difference'].max() != 0:
    raise RuntimeError('Los PNG almacenados no coinciden exactamente con la transformación regenerada.')

print('Auditoría píxel por píxel: CORRECTA')

## 5. Figura para R3

La línea gris representa la saturación ya presente en los JPEG originales. La línea azul representa la saturación total después del ajuste de brillo y la línea naranja muestra únicamente los píxeles que se saturaron por efecto del aumento de brillo.

In [ ]:
order = ['low', 'medium', 'high']
plot_data = summary.set_index('level').loc[order].reset_index()

plt.figure(figsize=(7.2, 4.5))
plt.plot(
    plot_data['brightness_factor'],
    plot_data['preexisting_any255_percent_global'],
    marker='o', linewidth=2, color='#777777', label='Pre-existing'
)
plt.plot(
    plot_data['brightness_factor'],
    plot_data['after_any255_percent_global'],
    marker='o', linewidth=2, color='#0072B2', label='After brightness adjustment'
)
plt.plot(
    plot_data['brightness_factor'],
    plot_data['newly_any255_percent_global'],
    marker='o', linewidth=2, color='#D55E00', label='Newly clipped'
)
plt.xlabel('Brightness factor')
plt.ylabel('Pixels with at least one RGB channel at 255 (%)')
plt.xticks(plot_data['brightness_factor'])
plt.grid(axis='y', alpha=0.25)
plt.legend(frameon=False)
plt.tight_layout()

figure_path = R3_ROOT / 'Figure_R3_brightness_clipping.png'
plt.savefig(figure_path, dpi=300, bbox_inches='tight')
plt.show()
print('Figura:', figure_path)

## 6. Auditoría del orden y la relación de aspecto

El cuaderno original de transformaciones trabajó sobre las imágenes RGB de 512×384. Las transformaciones se aplicaron antes del preprocesamiento de VGG-16 y se guardaron como PNG de 512×384. Después, el cuaderno de embeddings aplicó `Resize((224, 224))` tanto al original como a su contraparte transformada. Por ello, la relación 4:3 no se preservó, aunque la misma deformación geométrica se aplicó a ambos miembros de cada par.

In [ ]:
preprocessing_audit = pd.DataFrame([
    {
        'stage': 'Original files',
        'dimensions': '512 x 384',
        'aspect_ratio': '4:3 (1.333)',
        'operation': 'Load original JPEG and convert to RGB',
        'aspect_ratio_preserved': True,
    },
    {
        'stage': 'Controlled transformation',
        'dimensions': '512 x 384',
        'aspect_ratio': '4:3 (1.333)',
        'operation': 'Brightness/contrast; rotation with bilinear interpolation, expand=False, fill=0; or center crop followed by bilinear resize',
        'aspect_ratio_preserved': True,
    },
    {
        'stage': 'Stored transformed file',
        'dimensions': '512 x 384',
        'aspect_ratio': '4:3 (1.333)',
        'operation': 'Save transformed image as PNG',
        'aspect_ratio_preserved': True,
    },
    {
        'stage': 'VGG-16 input',
        'dimensions': '224 x 224',
        'aspect_ratio': '1:1 (1.000)',
        'operation': 'Direct Resize((224, 224)), ToTensor, ImageNet normalization; no center crop',
        'aspect_ratio_preserved': False,
    },
])
preprocessing_audit

## 7. Guardado de resultados

Comparte el ZIP generado para preparar los valores definitivos, la redacción del manuscrito y la respuesta punto por punto de R3.

In [ ]:
individual_path = R3_ROOT / '01_brightness_clipping_by_image.csv'
summary_path = R3_ROOT / '02_brightness_clipping_summary.csv'
preprocessing_path = R3_ROOT / '03_preprocessing_order_audit.csv'
environment_path = R3_ROOT / 'entorno_R3.json'

individual.to_csv(individual_path, index=False)
summary.to_csv(summary_path, index=False)
preprocessing_audit.to_csv(preprocessing_path, index=False)

environment = {
    'seed': SEED,
    'python': platform.python_version(),
    'pillow': PIL.__version__,
    'torch': torch.__version__,
    'torchvision': torchvision.__version__,
    'brightness_factors': BRIGHTNESS,
    'total_original_images': 2527,
    'original_dimensions': [512, 384],
    'transformation_order': 'load 512x384 RGB -> apply transformation -> save 512x384 PNG -> direct Resize((224,224)) -> tensor -> ImageNet normalization',
    'aspect_ratio_preserved_at_model_input': False,
    'clipping_primary_definition': 'pixel with at least one RGB channel equal to 255',
}
environment_path.write_text(json.dumps(environment, indent=2), encoding='utf-8')

archive_base = BASE_DIR / 'Revision_R3_Confounds_VGG16'
archive_path = Path(str(archive_base) + '.zip')
if archive_path.exists():
    archive_path.unlink()

import shutil
shutil.make_archive(str(archive_base), 'zip', root_dir=R3_ROOT)

print('Guardados:')
for path in [individual_path, summary_path, preprocessing_path, figure_path, environment_path, archive_path]:
    print('-', path)
print('\nComparte este archivo:', archive_path)

## Cómo interpretar el resultado

- Si `newly_any255_percent_global` aumenta con 1.2, 1.4 y 1.6, el ajuste de brillo introdujo clipping creciente.
- Esto permite afirmar que la condición de brillo no representa únicamente una variación lineal de luminancia: también incluye pérdida por saturación en parte de los píxeles.
- La saturación puede contribuir al perfil representacional observado, pero **no debe declararse como su causa** sin un experimento específico que controle el clipping.
- Las etiquetas low, medium y high son específicas de cada transformación. Un brillo high de 1.6, una rotación high de 60° y un zoom high de 1.50× no son severidades físicamente equivalentes.